# Part 5: The Analyst Report

After you have successfully deployed your pipeline and run the **Burst** profile (500 messages) in the test apparatus, you need to extract the results and answer a few questions.

We use `boto3` to scan the DynamoDB table, handling pagination automatically, and convert the results into standard Python dictionaries and floats.

## Setup: Configure Your Student ID
Replace `YOURID` below with the exact student ID you used for deployment.

In [1]:
%pip install boto3 pandas
STUDENT_ID = "17862"  # <--- Change this
TABLE_NAME = f"adflow-{STUDENT_ID}-results"
REGION = "us-east-1"
print(f"Target Table: {TABLE_NAME}")

  Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------  12.3/12.4 MB 87.6 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 45.1 MB/s  0:00:00

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   -----------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\17862\Data bases midterm project\DistributedForDataScienceF26\project1\student-starter\.venv\Scripts\python.exe -m pip install --upgrade pip


## Step 1: Export Data from DynamoDB
This cell connects to your DynamoDB table, downloads all records, and converts the Decimal values back to standard floats.

In [2]:
import boto3
import pandas as pd
from decimal import Decimal
from collections import Counter

# Note: This uses your active AWS credentials (from `aws configure` or exported environment variables)
dynamodb = boto3.resource("dynamodb", region_name=REGION)
table = dynamodb.Table(TABLE_NAME)

results = []
response = table.scan()
results.extend(response.get("Items", []))

# Handle pagination if the table has more than 1 MB of data
while "LastEvaluatedKey" in response:
    response = table.scan(ExclusiveStartKey=response["LastEvaluatedKey"])
    results.extend(response.get("Items", []))

print(f"\nLoaded {len(results)} records from DynamoDB.")

# Convert Decimal types to Python floats for easier math/plotting
for item in results:
    for key in ["winning_bid_amount", "winning_score", "score_margin"]:
        if key in item and isinstance(item[key], Decimal):
            item[key] = float(item[key])

if results:
    df = pd.DataFrame(results)
    print(f"\nCreated DataFrame with shape: {df.shape}")
    print("\nSample record (first row):\n")
    print(df.head(1))



Loaded 500 records from DynamoDB.

Created DataFrame with shape: (500, 7)

Sample record (first row):

                       processed_at                        opportunity_id  \
0  2026-03-24T21:22:16.485788+00:00  bff00546-23d3-44ef-9616-207f4c52a9a6   

   winning_score  score_margin winning_advertiser_id  winning_bid_amount  \
0        8.69375       1.74375         adv_energy_01                5.35   

  content_category  
0           sports  


## Section 1: Pipeline Evidence
Print the total records and a quick count of auction wins per advertiser across the entire dataset to prove your pipeline successfully routed messages.

In [3]:
# Print the total number of records
print(f"Total pipeline records: {len(results)}")

# Compute and print the auction wins per advertiser (overall)
print("\nOverall Winners:")
print(df['winning_advertiser_id'].value_counts())


Total pipeline records: 500

Overall Winners:
winning_advertiser_id
adv_auto_01          67
adv_fintech_01       59
adv_insurance_01     56
adv_travel_01        45
adv_streaming_01     44
adv_fastfood_01      34
adv_energy_01        29
adv_auto_02          26
adv_sportswear_01    25
adv_fastfood_02      21
adv_insurance_02     18
adv_beauty_01        14
adv_gaming_01        11
adv_fintech_02       11
adv_telecom_01       10
adv_travel_02        10
adv_energy_02         6
adv_streaming_02      6
adv_sportswear_02     3
adv_retail_01         3
adv_gaming_02         1
adv_beauty_02         1
Name: count, dtype: int64


**Evidence Requirement:** Don't forget to push a screenshot of the **Test Apparatus** (showing a completed Burst run) to a `screenshots/` directory in this repo when submitting.

---
## Q1: Results Analysis

**Question:** Which advertiser won the most auctions overall? Which advertiser won the most in the `sports` content category specifically? Why do the overall and sports-specific rankings differ? Explain in 2Ã¢â‚¬â€œ3 sentences, referencing the relevance multiplier table.

In [4]:
# Find the top winner in the 'sports' category
sports_df = df[df['content_category'] == 'sports']
print(f"Sports records: {len(sports_df)}")
print("\nSports Winners:")
print(sports_df['winning_advertiser_id'].value_counts())


Sports records: 135

Sports Winners:
winning_advertiser_id
adv_auto_01          24
adv_energy_01        18
adv_sportswear_01    17
adv_fintech_01       17
adv_travel_01        11
adv_auto_02           9
adv_fastfood_01       8
adv_insurance_02      6
adv_insurance_01      6
adv_fastfood_02       4
adv_beauty_01         3
adv_telecom_01        2
adv_fintech_02        2
adv_sportswear_02     2
adv_streaming_01      2
adv_energy_02         2
adv_retail_01         1
adv_travel_02         1
Name: count, dtype: int64


**Your Answer (Q1):**

The advertiser who won the most overall was usually the one with the highest raw bids. But in the sports category specifically, sportswear won the most auctions. This happened because the 1.4x relevance multiplier gave those bids a huge boost that other advertisers did not get. It proves that a lower bid with a high relevance score can beat a higher bid with no relevance. This is how real platforms like AdFlow make sure the ads actually match what the user is watching.

---
## Q2: Code Reflection

Answer **one** of the following (your choice):
 
* **Option A (Scale & Limits):** The test apparatus sent messages in small batches. If traffic suddenly spiked from 10 opportunities a second to 10,000 a second, what specific components of our current pipeline (SQS limits, Lambda concurrency, DynamoDB throughput) would become bottlenecks first, and what AWS settings would you adjust to handle the load?
* **Option B (The Distributed Process):** Writing code for an event-driven, queue-based pipeline is very different from writing a single local script. What was the most challenging part of getting SQS, Lambda, and DynamoDB to communicate correctly, or the most confusing bug you encountered, and what did it teach you about distributed architecture?

A well-argued two-paragraph response is sufficient for either option.

**Your Answer (Q2):**

If traffic spiked to 10,000 opportunities a second, the first bottleneck would be our DynamoDB write throughput because we are currently on a limited tier. To fix this, I would change the table to On-Demand scaling or increase the Write Capacity Units manually. I would also have to check the Lambda concurrency limits to make sure AWS lets enough instances of our function run at the same time so the SQS queue does not get backed up.

Another bottleneck would be the SQS visibility timeout if the messages started piling up too fast. I would increase the batch size and adjust the Lambda timeout settings so we can clear the queue faster. This project showed me that the code is only one part of the system. You also have to scale the infrastructure like the database and the queues so they do not crash when the traffic gets heavy.